# 11 - AG-UI Streaming
Implement the AG-UI protocol using the StrandsAgent adapter for SSE streaming, covering all 16 event types and frontend integration patterns.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import asyncio
from pathlib import Path

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
MODEL_ID = config["MODEL_ID"]
GRAPH_ID = config["NEPTUNE_GRAPH_ID"]

print(f"\u2713 Config loaded (model={MODEL_ID})")

## Step 2: Understand AG-UI Event Types

AG-UI defines 16 event types for streaming agent interactions to frontends.

In [ ]:
from ag_ui.core import RunAgentInput
from ag_ui.encoder import EventEncoder

# AG-UI Event Types
event_types = [
    ("RUN_STARTED", "Agent run has begun"),
    ("RUN_FINISHED", "Agent run completed"),
    ("RUN_ERROR", "Agent run encountered an error"),
    ("TEXT_MESSAGE_START", "Beginning of a text message"),
    ("TEXT_MESSAGE_CONTENT", "Streaming text content chunk"),
    ("TEXT_MESSAGE_END", "End of a text message"),
    ("TOOL_CALL_START", "Agent is calling a tool"),
    ("TOOL_CALL_ARGS", "Streaming tool call arguments"),
    ("TOOL_CALL_END", "Tool call completed"),
    ("TOOL_CALL_RESULT", "Result from tool execution"),
    ("STATE_SNAPSHOT", "Full state snapshot for sync"),
    ("STATE_DELTA", "Incremental state update"),
    ("MESSAGES_SNAPSHOT", "Full message history snapshot"),
    ("CUSTOM", "Custom application-specific events"),
    ("RAW", "Raw/passthrough events"),
    ("STEP", "Agent reasoning step"),
]

print("AG-UI Event Types:")
for event_type, description in event_types:
    print(f"  {event_type:25s} - {description}")

print(f"\n\u2713 {len(event_types)} event types available")

## Step 3: Create StrandsAgent with AG-UI Adapter

The `ag_ui_strands` package adapts Strands agent output to AG-UI events.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from ag_ui_strands import StrandsAgent, StrandsAgentConfig

neptune = boto3.client("neptune-graph", region_name=REGION)

@tool
def search_experts(topic: str) -> str:
    """Search knowledge graph for experts on a topic.
    
    Args:
        topic: The expertise topic to search for
    """
    query = """
    MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic)
    WHERE toLower(t.name) CONTAINS toLower($topic)
    RETURN p.name AS name, p.title AS title, r.strength AS strength
    ORDER BY r.strength DESC LIMIT 3
    """
    response = neptune.execute_query(
        graphIdentifier=GRAPH_ID,
        queryString=query,
        language="OPEN_CYPHER",
        parameters={"topic": {"value": topic}}
    )
    results = json.loads(response["payload"].read())
    output = []
    for row in results["results"]:
        output.append(f"{row['name']} ({row['title']}) - strength: {row['strength']}")
    return "\n".join(output) if output else "No experts found"

# Create the base Strands agent
model = BedrockModel(model_id=MODEL_ID, region_name=REGION)
base_agent = Agent(
    model=model,
    system_prompt="You are OrgIQ Search. Help users find expertise in the organization.",
    tools=[search_experts]
)

print("\u2713 Base Strands agent created")

## Step 4: Wrap with AG-UI StrandsAgent Adapter

In [ ]:
# Create the AG-UI adapter
ag_ui_agent = StrandsAgent(
    agent=base_agent,
    config=StrandsAgentConfig(
        agent_id=f"{PREFIX}-search",
        emit_tool_calls=True,
        emit_state_events=True,
        emit_step_events=True
    )
)

print("\u2713 AG-UI StrandsAgent adapter created")
print(f"  Agent ID: {PREFIX}-search")
print(f"  Tool calls: emitted")
print(f"  State events: emitted")
print(f"  Step events: emitted")

## Step 5: Generate AG-UI Events

Run the agent and capture the AG-UI event stream.

In [ ]:
# Create a RunAgentInput (what the frontend sends)
run_input = RunAgentInput(
    thread_id="thread-001",
    run_id="run-001",
    messages=[
        {"role": "user", "content": "Who are the top Kubernetes experts?"}
    ]
)

# Collect events
events = []
async def collect_events():
    async for event in ag_ui_agent.run(run_input):
        events.append(event)
    return events

# Run and collect
await collect_events()

print(f"\u2713 Collected {len(events)} AG-UI events")
print("\nEvent sequence:")
for i, event in enumerate(events):
    event_type = event.type if hasattr(event, 'type') else type(event).__name__
    print(f"  {i+1:2d}. {event_type}")

## Step 6: Inspect Event Details

In [ ]:
# Show details of key events
print("Event details:\n")
for event in events:
    event_type = event.type if hasattr(event, 'type') else type(event).__name__
    
    if event_type == "TEXT_MESSAGE_CONTENT":
        content = getattr(event, 'content', getattr(event, 'delta', ''))[:60]
        print(f"  TEXT_MESSAGE_CONTENT: \"{content}...\"")
    elif event_type == "TOOL_CALL_START":
        tool_name = getattr(event, 'name', getattr(event, 'tool_name', 'unknown'))
        print(f"  TOOL_CALL_START: {tool_name}")
    elif event_type == "TOOL_CALL_RESULT":
        result_preview = str(getattr(event, 'result', ''))[:80]
        print(f"  TOOL_CALL_RESULT: {result_preview}...")
    elif event_type in ["RUN_STARTED", "RUN_FINISHED", "RUN_ERROR"]:
        print(f"  {event_type}")

print("\n\u2713 Event inspection complete")

## Step 7: Encode Events as SSE

Convert AG-UI events to Server-Sent Events format for HTTP streaming.

In [ ]:
# Encode events as SSE (what gets sent over HTTP)
encoder = EventEncoder()

print("SSE output (first 5 events):\n")
for event in events[:5]:
    sse_data = encoder.encode(event)
    print(sse_data)
    print("---")

print("\u2713 SSE encoding working")

## Step 8: Build HTTP Endpoint Handler

Show how to serve AG-UI events via an HTTP POST endpoint (as required by AG-UI spec).

In [ ]:
# AG-UI uses POST for SSE (not GET EventSource)
# This is the handler pattern for the middleware

from typing import AsyncGenerator

async def ag_ui_handler(request_body: dict) -> AsyncGenerator[str, None]:
    """HTTP POST handler that streams AG-UI events as SSE.
    
    Frontend calls: POST /api/agent/run with RunAgentInput body
    Response: text/event-stream (SSE)
    """
    run_input = RunAgentInput(**request_body)
    encoder = EventEncoder()
    
    async for event in ag_ui_agent.run(run_input):
        yield encoder.encode(event)

# Simulate the handler
request = {
    "thread_id": "thread-002",
    "run_id": "run-002",
    "messages": [{"role": "user", "content": "Find security experts"}]
}

print("Simulating AG-UI HTTP handler:")
print(f"  POST /api/agent/run")
print(f"  Content-Type: application/json")
print(f"  Response: text/event-stream\n")

event_count = 0
async for sse_chunk in ag_ui_handler(request):
    event_count += 1
    if event_count <= 3:
        print(f"  chunk {event_count}: {sse_chunk[:80]}...")

print(f"\n  Total chunks: {event_count}")
print("\u2713 HTTP handler pattern working")

## Step 9: State Events for Frontend Sync

In [ ]:
# AG-UI state events allow the frontend to stay in sync
# State snapshots contain the full agent state
# State deltas contain incremental updates

state_events = [e for e in events if hasattr(e, 'type') and 'STATE' in str(getattr(e, 'type', ''))]

print(f"State events in stream: {len(state_events)}")
for event in state_events:
    event_type = event.type if hasattr(event, 'type') else type(event).__name__
    if hasattr(event, 'state'):
        state_keys = list(event.state.keys()) if isinstance(event.state, dict) else str(event.state)[:50]
        print(f"  {event_type}: keys={state_keys}")
    else:
        print(f"  {event_type}")

print("\n\u2713 State events enable frontend synchronization")

## Step 10: Custom Events

In [ ]:
# Custom events for application-specific data
# Example: Send graph visualization data to frontend

from ag_ui.core import CustomEvent

# Create custom events for frontend features
custom_events = [
    CustomEvent(
        name="graph_update",
        data={
            "nodes": [{"id": "alice", "label": "Alice Chen", "type": "Person"}],
            "edges": [{"source": "alice", "target": "k8s", "label": "KNOWS_ABOUT"}]
        }
    ),
    CustomEvent(
        name="expertise_scores",
        data={
            "topic": "Kubernetes",
            "scores": [{"person": "Alice Chen", "score": 0.9}, {"person": "Dave Park", "score": 0.85}]
        }
    ),
]

encoder = EventEncoder()
print("Custom events (for @xyflow/react graph viz):")
for event in custom_events:
    sse = encoder.encode(event)
    print(f"  {sse[:100]}...")

print("\n\u2713 Custom events for frontend visualization")

## Step 11: Frontend Integration Pattern

Show how the React frontend consumes AG-UI events.

In [ ]:
# Frontend consumption pattern (TypeScript/React)
frontend_code = """
// AG-UI uses POST for SSE (not GET EventSource)
// Must use fetch + ReadableStream

async function* streamAgentResponse(input: RunAgentInput) {
  const response = await fetch('/api/agent/run', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(input),
  });

  const reader = response.body!.getReader();
  const decoder = new TextDecoder();
  let buffer = '';

  while (true) {
    const { done, value } = await reader.read();
    if (done) break;
    
    buffer += decoder.decode(value, { stream: true });
    const lines = buffer.split('\\n\\n');
    buffer = lines.pop() || '';
    
    for (const line of lines) {
      if (line.startsWith('data: ')) {
        const event = JSON.parse(line.slice(6));
        yield event;
      }
    }
  }
}
"""

print("Frontend AG-UI consumption pattern:")
print(frontend_code)
print("\u2713 Frontend pattern documented")
print("  Key: AG-UI uses POST (not GET EventSource)")
print("  Key: Must use fetch + ReadableStream")

## Summary

- [x] AG-UI 16 event types documented
- [x] StrandsAgent adapter wrapping base agent
- [x] Event stream generated from agent execution
- [x] Event inspection (text, tool calls, results)
- [x] SSE encoding via EventEncoder
- [x] HTTP POST handler pattern (not GET EventSource)
- [x] State events for frontend sync
- [x] Custom events for graph visualization
- [x] Frontend consumption pattern documented

**Next:** [12-a2a-protocol.ipynb](12-a2a-protocol.ipynb) — Agent-to-agent communication with A2A protocol